# 01.07 - Introducción a Merge/Join: Conectando Tablas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-01-fundamentos/notebooks/01_07_introduccion_merge_join.ipynb)

## Objetivos de Aprendizaje

Al finalizar este notebook, usted será capaz de:

1. **Comprender** por qué los datos se almacenan en múltiples tablas
2. **Identificar** claves primarias y claves foráneas en conjuntos de datos
3. **Distinguir** los tipos de uniones (inner, left, right, outer)
4. **Anticipar** problemas comunes al unir tablas

## Duración estimada: 40 minutos

---

## 1. ¿Por qué Múltiples Tablas?

> *"Los datos del mundo real rara vez viven en una sola tabla."*

### 1.1 El Problema de la Tabla Única

Imagine que queremos almacenar información de estudiantes y sus notas. Una tabla única se vería así:

| id_est | nombre | carrera | director_carrera | telefono_director | materia | nota | profesor | email_prof |
|--------|--------|---------|------------------|-------------------|---------|------|----------|------------|
| 001 | Ana | Admin | Dr. García | 555-1234 | Estadística | 15 | López | lopez@ucab.edu |
| 001 | Ana | Admin | Dr. García | 555-1234 | Finanzas | 17 | Martínez | martinez@ucab.edu |
| 002 | Juan | Econ | Dra. Pérez | 555-5678 | Estadística | 14 | López | lopez@ucab.edu |

### 1.2 Problemas de este enfoque

| Problema | Descripción | Ejemplo |
|----------|-------------|----------|
| **Redundancia** | Información repetida | "Dr. García" aparece en cada fila de Admin |
| **Inconsistencia** | Riesgo de datos contradictorios | Si cambia el teléfono, hay que actualizar muchas filas |
| **Espacio** | Almacenamiento innecesario | Mismos datos ocupan espacio múltiples veces |
| **Mantenimiento** | Difícil de actualizar | Cambiar email de profesor requiere muchas actualizaciones |

### 1.3 La Solución: Normalización

La **normalización** divide los datos en tablas relacionadas:

**Tabla: Estudiantes**
| id_estudiante | nombre | id_carrera |
|---------------|--------|------------|
| 001 | Ana | C01 |
| 002 | Juan | C02 |

**Tabla: Carreras**
| id_carrera | nombre_carrera | director | telefono |
|------------|----------------|----------|----------|
| C01 | Administración | Dr. García | 555-1234 |
| C02 | Economía | Dra. Pérez | 555-5678 |

**Tabla: Notas**
| id_estudiante | id_materia | nota |
|---------------|------------|------|
| 001 | M01 | 15 |
| 001 | M02 | 17 |
| 002 | M01 | 14 |

**Tabla: Materias**
| id_materia | nombre_materia | id_profesor |
|------------|----------------|-------------|
| M01 | Estadística | P01 |
| M02 | Finanzas | P02 |

**Ventajas:**
- Cada dato se almacena una sola vez
- Actualizaciones en un solo lugar
- Menor riesgo de inconsistencias

---

## 2. Claves: El Pegamento de las Tablas

### 2.1 Clave Primaria (Primary Key)

Es el **identificador único** de cada fila en una tabla.

**Características:**
- Único: No puede repetirse
- No nulo: Siempre debe tener valor
- Estable: No debería cambiar

**Ejemplos:**
- `id_estudiante` en tabla Estudiantes
- `id_carrera` en tabla Carreras
- Cédula de identidad (en teoría)
- Código de producto

### 2.2 Clave Foránea (Foreign Key)

Es una columna que **referencia** la clave primaria de otra tabla.

```
Tabla Estudiantes                    Tabla Carreras
┌──────────────┬─────────────┐      ┌─────────────┬─────────────┐
│id_estudiante │ id_carrera  │──────│ id_carrera  │ nombre      │
├──────────────┼─────────────┤      ├─────────────┼─────────────┤
│ 001          │ C01 ───────────────│ C01         │ Admin       │
│ 002          │ C02 ───────────────│ C02         │ Economía    │
└──────────────┴─────────────┘      └─────────────┴─────────────┘
     Clave           Clave                Clave
    Primaria       Foránea              Primaria
```

### 2.3 Ejercicio de Identificación

En el siguiente esquema, identifique las claves:

```
Tabla: Facturas
| num_factura | fecha | id_cliente | total |

Tabla: Clientes
| id_cliente | nombre | email |

Tabla: Detalle_Factura
| num_factura | id_producto | cantidad | precio_unitario |

Tabla: Productos
| id_producto | descripcion | categoria |
```

**Respuestas:**
- Claves primarias: ___
- Claves foráneas: ___

### Solución del Ejercicio

**Claves primarias:**
- `Facturas`: `num_factura`
- `Clientes`: `id_cliente`
- `Detalle_Factura`: Combinación de `num_factura` + `id_producto` (clave compuesta)
- `Productos`: `id_producto`

**Claves foráneas:**
- `Facturas.id_cliente` → referencia a `Clientes.id_cliente`
- `Detalle_Factura.num_factura` → referencia a `Facturas.num_factura`
- `Detalle_Factura.id_producto` → referencia a `Productos.id_producto`

---

## 3. Tipos de Uniones (Joins)

Las uniones permiten **combinar** información de múltiples tablas basándose en una columna común.

### Ejemplo Base

Usaremos estas dos tablas simples:

**Tabla A: Estudiantes**
| id | nombre |
|----|--------|
| 1 | Ana |
| 2 | Juan |
| 3 | María |

**Tabla B: Notas**
| id_est | nota |
|--------|------|
| 1 | 15 |
| 2 | 18 |
| 4 | 12 |

Note que:
- María (id=3) no tiene nota registrada
- Hay una nota para id=4, pero no existe ese estudiante

### 3.1 INNER JOIN: Solo Coincidencias

Devuelve **solo las filas que tienen coincidencia en ambas tablas**.

```
    Estudiantes          Notas              RESULTADO
    ┌────┬───────┐     ┌────────┬──────┐    ┌────┬───────┬──────┐
    │ id │nombre │     │id_est  │ nota │    │ id │nombre │ nota │
    ├────┼───────┤     ├────────┼──────┤    ├────┼───────┼──────┤
    │ 1  │ Ana   │────│ 1      │ 15   │───│ 1  │ Ana   │ 15   │
    │ 2  │ Juan  │────│ 2      │ 18   │───│ 2  │ Juan  │ 18   │
    │ 3  │ María │ ✗   │ 4      │ 12   │ ✗  └────┴───────┴──────┘
    └────┴───────┘     └────────┴──────┘
```

**María no aparece** (no tiene nota)
**La nota de id=4 no aparece** (no existe el estudiante)

**Cuándo usar:** Cuando solo necesita registros con información completa en ambas tablas.

### 3.2 LEFT JOIN: Todas las Filas de la Izquierda

Devuelve **todas las filas de la tabla izquierda**, y las coincidencias de la derecha (NULL si no hay).

```
    Estudiantes          Notas              RESULTADO
    (IZQUIERDA)         (DERECHA)
    ┌────┬───────┐     ┌────────┬──────┐    ┌────┬───────┬──────┐
    │ id │nombre │     │id_est  │ nota │    │ id │nombre │ nota │
    ├────┼───────┤     ├────────┼──────┤    ├────┼───────┼──────┤
    │ 1  │ Ana   │────│ 1      │ 15   │───│ 1  │ Ana   │ 15   │
    │ 2  │ Juan  │────│ 2      │ 18   │───│ 2  │ Juan  │ 18   │
    │ 3  │ María │─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─│ 3  │ María │ NULL │
    └────┴───────┘     │ 4      │ 12   │ ✗  └────┴───────┴──────┘
                       └────────┴──────┘
```

**María aparece** con nota NULL
**La nota de id=4 no aparece** (no hay estudiante correspondiente)

**Cuándo usar:** Cuando necesita todos los registros de la tabla principal, aunque no tengan información relacionada.

### 3.3 RIGHT JOIN: Todas las Filas de la Derecha

Devuelve **todas las filas de la tabla derecha**, y las coincidencias de la izquierda.

```
    Estudiantes          Notas              RESULTADO
    (IZQUIERDA)         (DERECHA)
    ┌────┬───────┐     ┌────────┬──────┐    ┌────┬───────┬──────┐
    │ id │nombre │     │id_est  │ nota │    │ id │nombre │ nota │
    ├────┼───────┤     ├────────┼──────┤    ├────┼───────┼──────┤
    │ 1  │ Ana   │────│ 1      │ 15   │───│ 1  │ Ana   │ 15   │
    │ 2  │ Juan  │────│ 2      │ 18   │───│ 2  │ Juan  │ 18   │
    │ 3  │ María │ ✗   │ 4      │ 12   │───│NULL│ NULL  │ 12   │
    └────┴───────┘     └────────┴──────┘    └────┴───────┴──────┘
```

**María no aparece** (no tiene nota)
**La nota de id=4 aparece** con nombre NULL

**Cuándo usar:** Similar a LEFT pero priorizando la tabla secundaria. Es menos común.

### 3.4 FULL OUTER JOIN: Todas las Filas de Ambas

Devuelve **todas las filas de ambas tablas**, con NULL donde no hay coincidencia.

```
    Estudiantes          Notas              RESULTADO
    ┌────┬───────┐     ┌────────┬──────┐    ┌────┬───────┬──────┐
    │ id │nombre │     │id_est  │ nota │    │ id │nombre │ nota │
    ├────┼───────┤     ├────────┼──────┤    ├────┼───────┼──────┤
    │ 1  │ Ana   │────│ 1      │ 15   │───│ 1  │ Ana   │ 15   │
    │ 2  │ Juan  │────│ 2      │ 18   │───│ 2  │ Juan  │ 18   │
    │ 3  │ María │─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─│ 3  │ María │ NULL │
    └────┴───────┘     │ 4      │ 12   │───│ 4  │ NULL  │ 12   │
                       └────────┴──────┘    └────┴───────┴──────┘
```

**María aparece** con nota NULL
**La nota de id=4 aparece** con nombre NULL

**Cuándo usar:** Para identificar registros huérfanos en ambas tablas (útil en auditorías de datos).

### Resumen Visual de Joins

```
                    INNER JOIN            LEFT JOIN
                    ┌───────┐             ┌───────┐
                   │  ██   │            │███████│
                   │ █████ │            │███████│
                   │  ██   │            │ █████ │
                    └───────┘             └───────┘
                  Solo intersección      Todo de izquierda

                    RIGHT JOIN           FULL OUTER JOIN
                    ┌───────┐             ┌───────┐
                   │ █████ │            │███████│
                   │███████│            │███████│
                   │███████│            │███████│
                    └───────┘             └───────┘
                  Todo de derecha        Todo de ambas
```

---

## 4. Ejercicio Práctico

Dado el siguiente escenario, determine qué tipo de join usaría:

### Escenario 1
**Contexto:** Tiene una tabla de `empleados` y una tabla de `ventas`. Quiere un reporte de ventas **solo de empleados que hayan vendido algo**.

**Tipo de join:** ___

**Razonamiento:** ___

### Escenario 2
**Contexto:** Tiene una tabla de `estudiantes` y una tabla de `inscripciones`. Quiere listar **todos los estudiantes**, mostrando sus inscripciones si las tienen.

**Tipo de join:** ___

**Razonamiento:** ___

### Escenario 3
**Contexto:** Tiene una tabla de `productos` y una tabla de `inventario`. Quiere identificar **productos sin inventario** e **inventario sin producto correspondiente** (posibles errores de datos).

**Tipo de join:** ___

**Razonamiento:** ___

### Escenario 4
**Contexto:** Tiene una tabla de `clientes` y una tabla de `pedidos`. Quiere un reporte de **todos los pedidos** con información del cliente, incluso si el cliente fue eliminado.

**Tipo de join:** ___

**Razonamiento:** ___

### Soluciones

**Escenario 1:** INNER JOIN
- Solo queremos empleados con ventas (intersección)

**Escenario 2:** LEFT JOIN
- Queremos todos los estudiantes (tabla izquierda completa)
- Las inscripciones son opcionales

**Escenario 3:** FULL OUTER JOIN
- Necesitamos ver huérfanos de ambos lados
- Productos sin inventario Y inventario sin producto

**Escenario 4:** RIGHT JOIN (o LEFT con tablas invertidas)
- Pedidos es la tabla principal
- Queremos todos los pedidos aunque el cliente no exista

---

## 5. Problemas Comunes al Unir Tablas

### 5.1 Duplicación de Filas

**Problema:** Si la clave no es única, la unión multiplica las filas.

```
Estudiantes (1 fila)     Notas (3 filas con mismo id)
┌────┬───────┐           ┌────────┬──────┐
│ id │nombre │           │id_est  │ nota │
├────┼───────┤           ├────────┼──────┤
│ 1  │ Ana   │           │ 1      │ 15   │
└────┴───────┘           │ 1      │ 18   │
                         │ 1      │ 16   │
                         └────────┴──────┘

RESULTADO: 3 filas (una por cada nota de Ana)
┌────┬───────┬──────┐
│ id │nombre │ nota │
├────┼───────┼──────┤
│ 1  │ Ana   │ 15   │
│ 1  │ Ana   │ 18   │
│ 1  │ Ana   │ 16   │
└────┴───────┴──────┘
```

**Verificación:** Antes de unir, verifique que la clave sea única en al menos una tabla.

### 5.2 Tipos de Datos Incompatibles

**Problema:** La clave es `texto` en una tabla y `número` en otra.

```
Tabla A: id_cliente = "001" (texto)
Tabla B: id_cliente = 1 (número)
```

**Resultado:** No habrá coincidencias ("001" ≠ 1)

**Solución:** Convertir al mismo tipo antes de unir.

### 5.3 Valores Nulos en Claves

**Problema:** Las claves con NULL nunca coinciden (NULL ≠ NULL).

```
Tabla A: id = NULL
Tabla B: id = NULL
Resultado de INNER JOIN: No hay coincidencia
```

**Solución:** Limpiar nulos antes de unir, o usar COALESCE.

### 5.4 Nombres de Columnas Conflictivos

**Problema:** Ambas tablas tienen columna `nombre`.

**Resultado:** Ambigüedad en cuál `nombre` se refiere.

**Solución:** Usar alias de tabla (ej: `A.nombre`, `B.nombre`).

---

## 6. Aplicación en Contexto Universitario

### Caso: Sistema Académico FACES

Nuestro dataset de rendimiento académico proviene de varias tablas relacionadas:

```
┌─────────────────┐     ┌─────────────────┐
│   ESTUDIANTES   │     │    CARRERAS     │
├─────────────────┤     ├─────────────────┤
│ id_estudiante   │────│ id_carrera (PK) │
│ nombre          │     │ nombre_carrera  │
│ id_carrera (FK) │     │ director        │
│ semestre_actual │     │ facultad        │
└─────────────────┘     └─────────────────┘
         │
         │
┌────────┴────────┐
│   INSCRIPCIONES │
├─────────────────┤
│ id_estudiante(FK)│
│ id_materia (FK) │
│ periodo         │
│ nota_final      │
│ estado          │
└─────────────────┘
         │
         │
┌────────┴────────┐
│    MATERIAS     │
├─────────────────┤
│ id_materia (PK) │
│ nombre_materia  │
│ creditos        │
│ id_profesor(FK) │
└─────────────────┘
```

### Preguntas que requieren JOINs

1. **¿Cuántos estudiantes tiene cada carrera?**
   - JOIN entre Estudiantes y Carreras

2. **¿Cuál es el promedio de notas por profesor?**
   - JOIN entre Inscripciones, Materias y Profesores

3. **¿Qué estudiantes no tienen inscripciones este período?**
   - LEFT JOIN entre Estudiantes e Inscripciones, filtrando NULLs

---

## 7. Conexión con Módulos Posteriores

### En Módulo 2 (Data Wrangling)

Aprenderá a implementar estos conceptos en código:

**Python/Pandas:**
```python
# Inner join
pd.merge(estudiantes, notas, on='id_estudiante', how='inner')

# Left join
pd.merge(estudiantes, notas, on='id_estudiante', how='left')
```

**SQL:**
```sql
-- Inner join
SELECT * FROM estudiantes
INNER JOIN notas ON estudiantes.id = notas.id_est;

-- Left join
SELECT * FROM estudiantes
LEFT JOIN notas ON estudiantes.id = notas.id_est;
```

### En Módulo 3 (Business Intelligence)

**Power BI / Power Query:**
- Establecer relaciones en el modelo de datos
- Combinar consultas (Merge Queries)

### En Módulo 4 (Modelos Predictivos)

- Preparación de datasets para modelado
- Feature engineering requiere unir múltiples fuentes

---

## 8. Resumen y Puntos Clave

### Conceptos Fundamentales

1. **Normalización**: Los datos se dividen en tablas para evitar redundancia

2. **Claves**:
   - Primaria: Identificador único de cada fila
   - Foránea: Referencia a clave primaria de otra tabla

3. **Tipos de JOIN**:
   - **INNER**: Solo coincidencias mutuas
   - **LEFT**: Todo de izquierda + coincidencias
   - **RIGHT**: Todo de derecha + coincidencias
   - **FULL OUTER**: Todo de ambas tablas

4. **Cuidados**:
   - Verificar unicidad de claves
   - Compatibilidad de tipos de datos
   - Manejo de valores nulos

### Checklist antes de hacer un JOIN

- [ ] ¿Identificó la columna común (clave)?  
- [ ] ¿Los tipos de datos coinciden?  
- [ ] ¿La clave es única en al menos una tabla?  
- [ ] ¿Qué tipo de join necesita (inner, left, etc.)?  
- [ ] ¿Cuántas filas espera en el resultado?  

---

## Referencias

- Codd, E. F. (1970). "A Relational Model of Data for Large Shared Data Banks". *Communications of the ACM*.
- Date, C. J. (2003). *An Introduction to Database Systems* (8th ed.). Pearson.
- McKinney, W. (2017). *Python for Data Analysis* (2nd ed.). O'Reilly.

---

**Próximo paso**: En el Módulo 2, implementará estos conceptos con `pd.merge()` en Python y queries SQL.